In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.utils import shuffle

# 파일 경로 지정: 코랩 노트북과 같은 폴더에 creditcard.csv가 있다고 가정
csv_path_candidates = [
    Path("./creditcard.csv"),
    Path("/content/creditcard.csv"),   # 코랩 기본 경로
    Path("/mnt/data/creditcard.csv"),  # (참고) 업로드된 경로가 다를 수 있음
]

csv_path = None
for p in csv_path_candidates:
    if p.exists():
        csv_path = p
        break

if csv_path is None:
    raise FileNotFoundError("creditcard.csv를 찾을 수 없습니다. 파일을 업로드하거나 경로를 수정해주세요.")

print(f"읽는 파일: {csv_path}")
df = pd.read_csv(csv_path)

# 기본 확인
print(df.shape, df.columns[:10])
assert "Class" in df.columns, "데이터에 'Class' 컬럼이 없습니다."
df["Class"].value_counts(dropna=False)

RATIO_NORMAL_TO_FRAUD = 9  # 정상:사기 = 9:1

# 클래스 분리
df_normal = df[df["Class"] == 0]
df_fraud  = df[df["Class"] == 1]

n_fraud = len(df_fraud)
n_normal_target = RATIO_NORMAL_TO_FRAUD * n_fraud

print(f"원본 정상 개수: {len(df_normal):,}")
print(f"원본 사기 개수 : {n_fraud:,}")
print(f"목표 정상 개수(9:1): {n_normal_target:,}")

# 정상 다운샘플링 (대체샘플링 없이)
if n_normal_target <= len(df_normal):
    df_normal_down = df_normal.sample(n=n_normal_target, random_state=42, replace=False)
else:
    # 이 경우는 거의 없지만, 혹시 정상 표본이 목표보다 적다면 가능한 만큼만 사용
    print("[경고] 정상 표본이 목표 수보다 적어, 가능한 만큼만 사용합니다.")
    df_normal_down = df_normal.copy()

# 합치고 셔플
df_down = pd.concat([df_normal_down, df_fraud], axis=0)
df_down = shuffle(df_down, random_state=42).reset_index(drop=True)

print("\n다운샘플링 후 클래스 분포:")
print(df_down["Class"].value_counts())
print(df_down.shape)

out_path = Path("./creditcard_downsampled_9to1.csv")
df_down.to_csv(out_path, index=False)
print(f"저장 완료: {out_path.resolve()}")